# Clase 4 — Datos y Sensores del Dominio Pesquero
## 🟡 Nivel INTERMEDIO — Explorar y modificar el análisis

**Curso:** Inteligencia Artificial Aplicada a la Producción Pesquera
**Institución:** UTN FRCh · PesquerosEnIA · 2026
**Docentes:** Ariel Giamportone · Soraya Corvalán

---

**Para quién:** ya tenés algo de Python (leés código, cambiás parámetros). Este notebook
recorre el flujo completo de exploración de datos oceanográficos y de captura, e incluye
**ejercicios 🟡** para que modifiques el código y observes el efecto.

**Contenido:** SST · clorofila · AIS · registros de captura · integración ambiental ·
acceso a datos reales.

## 1. Setup e importación de librerías

Importamos las librerías estándar de ciencia de datos. Todas están disponibles en Google Colab sin instalación adicional.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)  # Reproducibilidad

print('✓ Librerías cargadas correctamente')
print(f'  NumPy {np.__version__} | Pandas {pd.__version__}')

## 2. Temperatura Superficial del Mar (SST)

La SST es la variable oceanográfica más relevante para la pesca. En la Plataforma Continental Argentina (PCA) conviven dos masas de agua:

- **Corriente de Malvinas:** agua fría (~5–10°C), fluye hacia el norte por el talud continental
- **Corriente de Brasil:** agua cálida (~18–24°C), fluye hacia el sur por la costa uruguaya
- **Zona de confluencia (~40°S):** mezcla de ambas corrientes, alta productividad biológica

Generamos un grid sintético que simula las condiciones reales de la PCA.

In [ ]:
# ── Parámetros geográficos de la PCA ──────────────────────────────────────────
# Latitudes: de Tierra del Fuego (-55°S) a Buenos Aires (-34°S)
# Longitudes: plataforma continental (-65°O a -44°O)
latitudes = np.arange(-55, -34, 0.5)
longitudes = np.arange(-65, -44, 0.5)
lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

# ── SST media anual: gradiente latitudinal realista ───────────────────────────
# Norte (~-35°S): ~18°C | Sur (~-55°S): ~5°C
# Pendiente: ~0.65°C por grado de latitud
sst_base = 18 + 0.65 * lat_grid

# Variabilidad espacial: efecto de frente Malvinas-Brasil cerca del talud
ruido_espacial = np.random.normal(0, 0.8, lon_grid.shape)
gradiente_lon = 0.05 * (lon_grid + 52)  # frente cerca de la isobata de 200m
sst_media_anual = sst_base + ruido_espacial + gradiente_lon

print(f'Grid generado: {lat_grid.shape[0]} latitudes × {lon_grid.shape[1]} longitudes')
print(f'SST media: {sst_media_anual.mean():.1f}°C | Mín: {sst_media_anual.min():.1f}°C | Máx: {sst_media_anual.max():.1f}°C')

In [ ]:
# ── Mapa de SST media anual ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

contour = ax.contourf(longitudes, latitudes, sst_media_anual,
                      levels=20, cmap='RdYlBu_r', alpha=0.9)
cbar = plt.colorbar(contour, ax=ax, shrink=0.8)
cbar.set_label('Temperatura (°C)', fontsize=12)

# Marcar puertos principales
puertos = {
    'Puerto Madryn': (-42.77, -65.03),
    'Mar del Plata': (-38.00, -57.53),
    'Rawson': (-43.30, -65.10),
    'Ushuaia': (-54.80, -68.30)
}
for nombre, (lat_p, lon_p) in puertos.items():
    ax.plot(lon_p, lat_p, 'k^', markersize=8)
    ax.annotate(nombre, (lon_p + 0.3, lat_p), fontsize=8, color='black')

# Línea aproximada del frente Malvinas-Brasil
ax.axhline(-40, linestyle='--', color='white', linewidth=1.5,
           label='Frente Malvinas-Brasil (~40°S)')

ax.set_xlabel('Longitud (°O)', fontsize=12)
ax.set_ylabel('Latitud (°S)', fontsize=12)
ax.set_title('Temperatura Superficial del Mar — Media Anual\nPlataforma Continental Argentina',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### 2.1. Variación estacional de la SST

La SST varía con las estaciones del año. En la zona de pesca de merluza (~42°S, frente a Puerto Madryn), el rango estacional es de aproximadamente 4°C entre verano (enero) e invierno (julio).

In [ ]:
# ── Serie temporal de SST en zona de pesca de merluza (~42°S) ─────────────────
dias = np.arange(365)
fechas = pd.date_range('2025-01-01', periods=365, freq='D')

# SST base para zona merluza (~42°S): ~10°C media + ciclo anual de amplitud 4°C
sst_merluza = 10 + 4 * np.sin(2 * np.pi * dias / 365 - np.pi / 2)
sst_merluza += np.random.normal(0, 0.5, 365)  # ruido realista

# SST para zona calamar (~44°S, algo más al sur)
sst_calamar = 9 + 3.5 * np.sin(2 * np.pi * dias / 365 - np.pi / 2)
sst_calamar += np.random.normal(0, 0.4, 365)

# Gráfico
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas, sst_merluza, color='steelblue', linewidth=1.5, label='Zona merluza (~42°S)')
ax.plot(fechas, sst_calamar, color='darkorange', linewidth=1.5, linestyle='--',
        label='Zona calamar (~44°S)')

# Sombrear rango óptimo de merluza (4-12°C)
ax.axhspan(4, 12, alpha=0.1, color='steelblue', label='Rango óptimo merluza (4-12°C)')

ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Temperatura Superficial del Mar (°C)', fontsize=12)
ax.set_title('Variación Estacional de la SST — Zonas de Pesca PCA (2025)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'SST media zona merluza: {sst_merluza.mean():.1f}°C')
print(f'SST mínima (invierno): {sst_merluza.min():.1f}°C | SST máxima (verano): {sst_merluza.max():.1f}°C')

## 3. Clorofila-a: indicador de productividad marina

La clorofila-a (Chl-a) mide la concentración de fitoplancton en la superficie del mar. Es el punto de partida de la cadena trófica marina:

> **Fitoplancton** (Chl-a alta) → Zooplancton → Anchoíta y juveniles → **Merluza y Calamar**

Las zonas de alta clorofila, especialmente en los **frentes oceanográficos**, suelen coincidir con áreas de mayor captura pesquera.

In [ ]:
# ── Clorofila-a: distribución espacial en la PCA ──────────────────────────────
np.random.seed(42)

# Alta productividad en la zona del frente (~-40°S) y golfos patagónicos
clorofila = np.abs(
    2.0 * np.exp(-((lat_grid + 40) ** 2) / 20)  # pico en el frente ~-40°S
    + 1.5 * np.exp(-((lat_grid + 43) ** 2) / 15)  # golfo San Jorge
    + 0.8 * np.exp(-((lat_grid + 47) ** 2) / 10)  # golfo San Matías
    + np.random.exponential(0.3, lon_grid.shape)   # ruido realista
)

# Figura comparativa: SST vs Clorofila
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SST
c1 = axes[0].contourf(longitudes, latitudes, sst_media_anual, levels=15, cmap='RdYlBu_r')
plt.colorbar(c1, ax=axes[0]).set_label('SST (°C)')
axes[0].set_title('Temperatura Superficial del Mar (SST)', fontweight='bold')
axes[0].set_xlabel('Longitud (°O)')
axes[0].set_ylabel('Latitud (°S)')

# Clorofila
c2 = axes[1].contourf(longitudes, latitudes, clorofila, levels=15, cmap='YlGn')
plt.colorbar(c2, ax=axes[1]).set_label('Clorofila-a (mg/m³)')
axes[1].set_title('Clorofila-a (productividad primaria)', fontweight='bold')
axes[1].set_xlabel('Longitud (°O)')
axes[1].set_ylabel('Latitud (°S)')

plt.suptitle('PCA: SST vs Clorofila-a — indicadores de zonas de pesca',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Relación SST - Clorofila ───────────────────────────────────────────────────
# Aplanar los grids para análisis
sst_flat = sst_media_anual.flatten()
chl_flat = clorofila.flatten()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sst_flat, chl_flat, alpha=0.3, s=8, color='teal')
ax.set_xlabel('Temperatura Superficial del Mar (°C)', fontsize=12)
ax.set_ylabel('Clorofila-a (mg/m³)', fontsize=12)
ax.set_title('Relación SST – Clorofila-a en la PCA\n(cada punto = un pixel del grid)',
             fontsize=12)

# Correlación
correlacion = np.corrcoef(sst_flat, chl_flat)[0, 1]
ax.text(0.05, 0.95, f'Correlación: {correlacion:.2f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

print(f'Coeficiente de correlación SST-Clorofila: {correlacion:.3f}')
print('Interpretación: alta clorofila tiende a concentrarse en zonas de temperatura intermedia')
print('(frentes oceanográficos: SST 8–14°C, enriquecimiento por afloramiento)')

### 🟡 Ejercicio 1 — Mover el frente productivo

En la celda de clorofila, el pico principal está centrado en **-40°S**
(`np.exp(-((lat_grid + 40) ** 2) / 20)`).

1. Cambiá el `40` por `38` y volvé a correr la celda.
2. ¿Hacia dónde se corrió la zona de mayor clorofila?
3. ¿Qué puerto quedaría mejor posicionado para esa zona?

*Pista: recordá que la latitud es negativa (sur).*

## 4. Datos AIS: el rastro digital de un barco pesquero

El sistema AIS (Automatic Identification System) transmite la posición, velocidad y datos de identidad de los barcos cada pocos segundos.

**Clave:** la **velocidad sobre el fondo (SOG)** permite inferir la actividad del barco:
- SOG < 4 nudos → probable arrastre o pesca activa
- SOG 4–7 nudos → tránsito entre lances
- SOG > 7 nudos → navegación de/hacia puerto

In [ ]:
# ── Dataset AIS simulado: B/P Patagónico — viaje de 7 días ───────────────────
np.random.seed(123)
n_registros = 240  # transmisiones cada 30 minutos = 5 días en zona
fechas_ais = pd.date_range('2026-01-15 06:00', periods=n_registros, freq='30min')

# Trayectoria: sale de Puerto Madryn, va a zona de pesca, vuelve
# Fase 1: navegación hacia zona (0-20h)
# Fase 2: pesca activa (20-150h)
# Fase 3: regreso a puerto (150-168h)
horas = np.arange(n_registros) * 0.5

# Posiciones simuladas
lat_ais = -42.77 + np.cumsum(np.where(horas < 10, -0.08,
                  np.where(horas < 75, np.random.normal(0, 0.04, n_registros),
                           0.07)))
lon_ais = -65.03 + np.cumsum(np.where(horas < 10, 0.12,
                  np.where(horas < 75, np.random.normal(0, 0.04, n_registros),
                           -0.10)))

# Velocidades por fase
velocidad_kn = np.where(
    horas < 10, np.random.normal(10, 1, n_registros),      # navegando
    np.where(horas < 75, np.abs(np.random.normal(3, 1, n_registros)),  # pescando
             np.random.normal(10.5, 0.8, n_registros)))    # regresando
velocidad_kn = np.clip(velocidad_kn, 0.5, 13)

# Clasificación de actividad
actividad = pd.cut(velocidad_kn,
                   bins=[0, 4.0, 7.0, 20],
                   labels=['pescando', 'transitando', 'navegando'])

datos_ais = pd.DataFrame({
    'timestamp': fechas_ais,
    'mmsi': '701234567',
    'nombre_barco': 'BP PATAGÓNICO I',
    'latitud': lat_ais[:n_registros],
    'longitud': lon_ais[:n_registros],
    'velocidad_kn': velocidad_kn,
    'actividad': actividad
})

print(f'Registros AIS generados: {len(datos_ais)}')
print('\nDistribución de actividad:')
print(datos_ais['actividad'].value_counts())
datos_ais.head()

In [ ]:
# ── Visualización: trayectoria + velocidad ────────────────────────────────────
colores_actividad = {'pescando': '#2196F3', 'transitando': '#FF9800', 'navegando': '#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Mapa de trayectoria coloreada por actividad
for actividad_tipo, color in colores_actividad.items():
    mask = datos_ais['actividad'] == actividad_tipo
    axes[0].scatter(datos_ais[mask]['longitud'], datos_ais[mask]['latitud'],
                    c=color, s=12, alpha=0.7, label=actividad_tipo.capitalize())

# Marcar inicio y fin
axes[0].plot(datos_ais['longitud'].iloc[0], datos_ais['latitud'].iloc[0],
             'gs', markersize=12, label='Puerto Madryn (salida/llegada)')
axes[0].set_xlabel('Longitud (°O)', fontsize=11)
axes[0].set_ylabel('Latitud (°S)', fontsize=11)
axes[0].set_title('Trayectoria del B/P Patagónico I\n(coloreada por actividad inferida)',
                  fontsize=12)
axes[0].legend(fontsize=9)

# Velocidad en el tiempo
axes[1].plot(datos_ais['timestamp'], datos_ais['velocidad_kn'],
             color='steelblue', linewidth=1, alpha=0.8)
axes[1].axhspan(0, 4, alpha=0.15, color='blue', label='Zona de pesca (< 4 kn)')
axes[1].axhspan(7, 15, alpha=0.1, color='red', label='Navegación (> 7 kn)')
axes[1].set_xlabel('Fecha/Hora', fontsize=11)
axes[1].set_ylabel('Velocidad (nudos)', fontsize=11)
axes[1].set_title('Velocidad en el tiempo\n(proxy de actividad pesquera)', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. Registros de captura: los datos operativos del sector

Los **partes de pesca** registran qué se capturó, dónde y cuándo. Son la principal fuente de datos operativos del sector y, bien analizados, revelan patrones estacionales, espaciales y de eficiencia.

Generamos un dataset histórico de 500 mareas que simula registros reales.

In [ ]:
# ── Dataset de capturas históricas (partes de pesca simulados) ─────────────────
np.random.seed(42)
n_mareas = 500

# Variables oceanográficas al momento de la marea
sst_marea = np.random.normal(10, 3.5, n_mareas)   # SST en zona de pesca
chl_marea = np.abs(np.random.exponential(1.8, n_mareas))  # Clorofila
profundidad_m = np.random.uniform(60, 280, n_mareas)
mes = np.random.randint(1, 13, n_mareas)
lat_captura = np.random.uniform(-50, -39, n_mareas)
lon_captura = np.random.uniform(-62, -50, n_mareas)

# Captura de merluza (tn) — función de SST con óptimo en 8-12°C
prob_exito_sst = np.exp(-((sst_marea - 10) ** 2) / 18)
prob_exito_chl = np.clip(chl_marea / 4, 0, 1)
prob_exito_prof = np.exp(-((profundidad_m - 140) ** 2) / 6000)

captura_base = 60 * (0.45 * prob_exito_sst + 0.3 * prob_exito_chl + 0.25 * prob_exito_prof)
captura_merluza_tn = np.abs(captura_base + np.random.normal(0, 10, n_mareas))

registros_captura = pd.DataFrame({
    'mes': mes,
    'lat_captura': lat_captura,
    'lon_captura': lon_captura,
    'profundidad_m': profundidad_m,
    'sst_grados': sst_marea,
    'clorofila_mg_m3': chl_marea,
    'captura_merluza_tn': captura_merluza_tn
})

# Agregar estación
registros_captura['estacion'] = pd.cut(
    registros_captura['mes'],
    bins=[0, 3, 6, 9, 12],
    labels=['Verano', 'Otoño', 'Invierno', 'Primavera'])

print(f'Partes de pesca generados: {len(registros_captura)}')
print(f'Captura promedio por marea: {registros_captura["captura_merluza_tn"].mean():.1f} tn')
registros_captura.describe().round(2)

In [ ]:
# ── Análisis de capturas: estacionalidad y variables ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribución de captura
axes[0, 0].hist(registros_captura['captura_merluza_tn'], bins=30,
                color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(registros_captura['captura_merluza_tn'].mean(),
                   color='red', linestyle='--', label=f'Media: {registros_captura["captura_merluza_tn"].mean():.0f} tn')
axes[0, 0].set_xlabel('Captura por marea (tn)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].set_title('Distribución de captura de merluza por marea')
axes[0, 0].legend()

# 2. Captura por estación
captura_estacion = registros_captura.groupby('estacion', observed=True)['captura_merluza_tn'].mean()
colores_est = ['#FF8C00', '#8B4513', '#4169E1', '#228B22']
captura_estacion.plot(kind='bar', ax=axes[0, 1], color=colores_est, alpha=0.85, edgecolor='white')
axes[0, 1].set_xlabel('Estación')
axes[0, 1].set_ylabel('Captura media (tn)')
axes[0, 1].set_title('Captura promedio de merluza por estación')
axes[0, 1].tick_params(axis='x', rotation=0)

# 3. SST vs Captura
axes[1, 0].scatter(registros_captura['sst_grados'], registros_captura['captura_merluza_tn'],
                   alpha=0.3, s=20, color='teal')
axes[1, 0].set_xlabel('SST (°C)')
axes[1, 0].set_ylabel('Captura merluza (tn)')
axes[1, 0].set_title('Temperatura vs Captura de merluza')

# Línea de tendencia
z = np.polyfit(registros_captura['sst_grados'], registros_captura['captura_merluza_tn'], 2)
p = np.poly1d(z)
x_line = np.linspace(registros_captura['sst_grados'].min(), registros_captura['sst_grados'].max(), 100)
axes[1, 0].plot(x_line, p(x_line), 'r-', linewidth=2, label='Tendencia cuadrática')
axes[1, 0].axvspan(8, 12, alpha=0.1, color='blue', label='Rango óptimo merluza')
axes[1, 0].legend(fontsize=9)

# 4. Mapa de capturas
scatter = axes[1, 1].scatter(registros_captura['lon_captura'], registros_captura['lat_captura'],
                              c=registros_captura['captura_merluza_tn'],
                              cmap='YlOrRd', s=20, alpha=0.6)
plt.colorbar(scatter, ax=axes[1, 1]).set_label('Captura (tn)')
axes[1, 1].set_xlabel('Longitud (°O)')
axes[1, 1].set_ylabel('Latitud (°S)')
axes[1, 1].set_title('Distribución espacial de capturas')

plt.suptitle('Análisis de Registros de Captura — Merluza Hubbsi, PCA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Integración: variables ambientales × captura

Cruzamos los datos ambientales con los registros de captura para identificar las condiciones óptimas de pesca. Este análisis es la base del modelo predictivo de la **Clase 6**.

In [ ]:
# ── Correlación entre variables ambientales y captura ─────────────────────────
variables_num = ['sst_grados', 'clorofila_mg_m3', 'profundidad_m', 'captura_merluza_tn']
correlaciones = registros_captura[variables_num].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de correlación
sns.heatmap(correlaciones, annot=True, fmt='.2f', cmap='RdYlBu',
            center=0, ax=axes[0], square=True,
            xticklabels=['SST (°C)', 'Clorofila', 'Profundidad', 'Captura (tn)'],
            yticklabels=['SST (°C)', 'Clorofila', 'Profundidad', 'Captura (tn)'])
axes[0].set_title('Correlaciones entre variables\nambientales y captura', fontsize=12)

# SST óptima para captura (binned)
bins_sst = pd.cut(registros_captura['sst_grados'], bins=8)
captura_por_sst = registros_captura.groupby(bins_sst, observed=True)['captura_merluza_tn'].mean()
captura_por_sst.plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Rango de SST (°C)')
axes[1].set_ylabel('Captura promedio (tn)')
axes[1].set_title('Captura de merluza por rango de SST\n(óptimo ~8-12°C)', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nCorrelación SST - Captura:', correlaciones.loc['sst_grados', 'captura_merluza_tn'].round(3))
print('Correlación Clorofila - Captura:', correlaciones.loc['clorofila_mg_m3', 'captura_merluza_tn'].round(3))

### 🟡 Ejercicio 2 — ¿El óptimo de SST cambia por estación?

Usando `registros_captura`:

1. Filtrá solo las mareas de invierno: `registros_captura[registros_captura['estacion'] == 'Invierno']`.
2. Recalculá la correlación SST–captura sobre ese subconjunto.
3. Compará con la correlación global. ¿El agua "óptima" es la misma en invierno que en verano?

```python
inv = registros_captura[registros_captura['estacion'] == 'Invierno']
print(inv[['sst_grados', 'captura_merluza_tn']].corr().round(3))
```

In [ ]:
# ── Resumen estadístico de condiciones óptimas de pesca ───────────────────────
# Definir "buena marea": captura > percentil 75
percentil_75 = registros_captura['captura_merluza_tn'].quantile(0.75)
buenas_mareas = registros_captura[registros_captura['captura_merluza_tn'] >= percentil_75]
malas_mareas  = registros_captura[registros_captura['captura_merluza_tn'] <  percentil_75]

print(f'Umbral de "buena marea" (percentil 75): {percentil_75:.1f} tn')
print(f'Buenas mareas: {len(buenas_mareas)} ({len(buenas_mareas)/len(registros_captura)*100:.0f}%)')
print()
print('Condiciones promedio en buenas mareas vs. resto:')
print(pd.DataFrame({
    'Buenas mareas': buenas_mareas[['sst_grados', 'clorofila_mg_m3', 'profundidad_m']].mean(),
    'Resto': malas_mareas[['sst_grados', 'clorofila_mg_m3', 'profundidad_m']].mean()
}).round(2))

## 7. Acceso a datos reales — Código de referencia

El siguiente bloque muestra cómo acceder a datos **reales** de las plataformas abiertas. El código está comentado porque requiere credenciales y conexión a internet, pero es funcional para usar fuera del curso.

In [ ]:
# ==============================================================================
# ACCESO A DATOS REALES — Para usar fuera del curso (requiere internet)
# ==============================================================================

# ── Copernicus Marine Service (SST, Clorofila, Corrientes) ────────────────────
# 1. Registrarse gratis en: https://marine.copernicus.eu
# 2. Instalar: pip install copernicusmarine
#
# import copernicusmarine
# ds = copernicusmarine.open_dataset(
#     dataset_id="cmems_obs-sst_glo_phy_nrt_l4_P1D-m",
#     variables=["analysed_sst"],
#     minimum_latitude=-55, maximum_latitude=-34,
#     minimum_longitude=-65, maximum_longitude=-44,
#     start_datetime="2024-01-01", end_datetime="2024-12-31"
# )
# sst_real = ds["analysed_sst"].values - 273.15  # Kelvin a Celsius

# ── NOAA ERDDAP (sin credenciales, acceso libre) ──────────────────────────────
# pip install erddapy
#
# from erddapy import ERDDAP
# e = ERDDAP(server="https://coastwatch.pfeg.noaa.gov/erddap/")
# e.dataset_id = "erdMH1chla8day"
# e.constraints = {
#     "time>=": "2024-01-01",
#     "time<=": "2024-12-31",
#     "latitude>=": -55, "latitude<=": -34,
#     "longitude>=": -65, "longitude<=": -44
# }
# df_chl = e.to_pandas()

# ── Global Fishing Watch (requiere token gratuito) ────────────────────────────
# Registro en: https://globalfishingwatch.org/data-download/
# API documentada en: https://globalfishingwatch.org/our-apis/
#
# import requests
# headers = {"Authorization": "Bearer TU_TOKEN_GFW"}
# url = "https://gateway.api.globalfishingwatch.org/v3/events"
# params = {"vessels": "mmsi:701234567", "startDate": "2024-01-01", "endDate": "2024-12-31"}
# response = requests.get(url, headers=headers, params=params)
# datos_gfw = response.json()

print('Código de referencia para acceso a datos reales.')
print('Descomenta las secciones correspondientes para usar con credenciales reales.')
print()
print('Recursos:')
print('  Copernicus Marine: https://marine.copernicus.eu')
print('  NOAA ERDDAP:       https://coastwatch.pfeg.noaa.gov/erddap')
print('  Global Fishing Watch: https://globalfishingwatch.org/data-download')

## Síntesis de la clase

En este notebook exploramos los **tipos de datos fundamentales del sector pesquero**:

| Dato | Fuente | Uso principal |
|------|--------|---------------|
| SST (temperatura del mar) | Copernicus, NOAA | Predicción de zonas de pesca |
| Clorofila-a | Copernicus, NASA | Productividad primaria, distribución de recursos |
| AIS / VMS | GFW, SSPA | Monitoreo de flota, detección de actividad |
| Partes de pesca | SSPA, empresa | Captura histórica por zona y especie |

**El resultado clave:** identificamos que la SST en el rango **8–12°C** se asocia con las mejores capturas de merluza en la PCA. Esta relación, combinada con clorofila-a y profundidad, es exactamente lo que vamos a modelar en la **Clase 6** con algoritmos de Machine Learning.

---

## Para explorar más

- **Copernicus Marine Service:** https://marine.copernicus.eu
- **Global Fishing Watch Map:** https://globalfishingwatch.org/map
- **INIDEP:** https://www.inidep.edu.ar
- **Repo de Ariel (modelo hidrodinámico acuicultura):** https://github.com/arielgiamportone/Hydrodinamic_model_Aquaculture_nets
- **PesquerosEnIA:** https://github.com/PesquerosEnIA
- **xarray tutorial (datos NetCDF):** https://tutorial.xarray.dev
- **Kroodsma et al. 2018 (Science):** https://globalfishingwatch.org/research